<h2 style="color:#34495e;">Natural Language Processing – Task 1</h2>

<hr style="border:1px solid #ddd;">

**👤 Name:** Tila Muhammad  
**🆔 Roll No:** 25i-7601  
**🎓 Class:** MS-AI  
**👨‍🏫 Instructor:** Dr. Zohair Ahmed

### Required Libraries

In [1]:
import torch
from datasets import load_dataset
import pandas as pd
import numpy as np

<div style="border-left:6px solid #ff9800; background:#f0fff4; padding:18px; border-radius:10px">

### 🎬 IMDB Dataset (Hugging Face)

- **Total Samples:** 100,000 reviews  
- **Splits:**
  - **Train:** 25,000 labeled reviews  
  - **Test:** 25,000 labeled reviews  
  - **Unsupervised:** 50,000 reviews (for unsupervised tasks)
- **Features:** `text`, `label`  
- **Task:** Binary Sentiment Classification (0 = Negative, 1 = Positive)  

</div>

In [2]:
print("Loading IMDB Dataset...")
imdb = load_dataset("imdb")
print(imdb)

Loading IMDB Dataset...
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [3]:
# Converting them to DataFrames
df_train = pd.DataFrame(imdb['train'])

#df_test = pd.DataFrame(imdb['test'])
#df_unsupervised = pd.DataFrame(imdb['unsupervised'])

In [4]:
print(df_train.head(2), "\nShape:", df_train.shape)

#print(df_test.tail(2), "\nShape:", df_test.shape)
#print(df_unsupervised)

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0 
Shape: (25000, 2)


### Label Distribution

In [5]:
print("\nLabel Distribution (Train):", df_train['label'].value_counts())

#print("\nLabel Distribution (Test):", df_test['label'].value_counts())


Label Distribution (Train): label
0    12500
1    12500
Name: count, dtype: int64


### Sentence Lengths

In [6]:
df_train['sentence_len'] = df_train['text'].apply(lambda x: len(x.split()))

#df_test['sentence_len'] = df_test['text'].apply(lambda x: len(x.split()))

In [7]:
print(df_train.tail(4))

#print(df_test)

                                                    text  label  sentence_len
24996  I love this movie like no other. Another time ...      1           183
24997  This film and it's sequel Barry Mckenzie holds...      1           134
24998  'The Adventures Of Barry McKenzie' started lif...      1           717
24999  The story centers around Barry McKenzie who mu...      1            55


In [8]:
# Pick the review index to check
index = 24999

# Print the review text
print("Review Text:\n")
print(df_train.loc[index, 'text'])

# Print the counted number of words from that column
print("\nNumber of words:", df_train.loc[index, 'sentence_len'])

Review Text:

The story centers around Barry McKenzie who must go to England if he wishes to claim his inheritance. Being about the grossest Aussie shearer ever to set foot outside this great Nation of ours there is something of a culture clash and much fun and games ensue. The songs of Barry McKenzie(Barry Crocker) are highlights.

Number of words: 55


### Vocabulary Size

In [9]:
vocab = set()

for review in df_train['text']:
    vocab.update(review.lower().split()) 
print("Approximate Vocabulary Size (Train):", len(vocab))

Approximate Vocabulary Size (Train): 251637


<div style="border-left:6px solid #4caf50; background:#f0fff4; padding:18px; border-radius:10px">

### 📝 CoNLL-2003 Dataset (Hugging Face)

- **Total Samples:** 21,744 sentences  
- **Splits:**
  - **Train:** 14,041 sentences  
  - **Validation:** 3,250 sentences  
  - **Test:** 3,453 sentences
- **Features:** `id`, `tokens`, `pos_tags`, `chunk_tags`, `ner_tags`  
- **Task:** Named Entity Recognition (NER)  
</div>

In [10]:
print("Loading CoNLL-2003 Dataset...")
conll = load_dataset("conll2003", trust_remote_code=True)
print(conll)

Loading CoNLL-2003 Dataset...
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [11]:
#print(conll['train'][0])

# Converting them to DataFrames
df_train = pd.DataFrame(conll['train'])

#df_validation = pd.DataFrame(conll['validation'])
#df_test = pd.DataFrame(conll['test'])

In [12]:
print(df_train.head(2), "\nShape:", df_train.shape)

#print(df_validation.head(2), "\nShape:", df_validation.shape)
#print(df_test.tail(2), "\nShape:", df_test.shape)

  id                                             tokens  \
0  0  [EU, rejects, German, call, to, boycott, Briti...   
1  1                                 [Peter, Blackburn]   

                              pos_tags                           chunk_tags  \
0  [22, 42, 16, 21, 35, 37, 16, 21, 7]  [11, 21, 11, 12, 21, 22, 11, 12, 0]   
1                             [22, 22]                             [11, 12]   

                      ner_tags  
0  [3, 0, 7, 0, 0, 0, 7, 0, 0]  
1                       [1, 2]   
Shape: (14041, 5)


### Label Distribution

In [13]:
label_names = conll['train'].features['ner_tags'].feature.names

# Initialize an empty dictionary to store counts
label_distribution = {label: 0 for label in label_names}

# Count labels manually
for example in conll['train']:
    for tag_id in example['ner_tags']:
        label = label_names[tag_id]
        label_distribution[label] += 1

# Print the distribution
print("Label Distribution in Training Set:")
for label, count in label_distribution.items():
    print(f"{label}: {count}")

Label Distribution in Training Set:
O: 169578
B-PER: 6600
I-PER: 4528
B-ORG: 6321
I-ORG: 3704
B-LOC: 7140
I-LOC: 1157
B-MISC: 3438
I-MISC: 1155


### Sentence Lengths

In [14]:
df_train['sentence_len'] = df_train['tokens'].apply(lambda x: len(x))
print(df_train.tail(4))

          id                     tokens          pos_tags        chunk_tags  \
14037  14037            [Division, two]          [21, 11]          [11, 12]   
14038  14038  [Plymouth, 2, Preston, 1]  [21, 11, 22, 11]  [11, 12, 12, 12]   
14039  14039          [Division, three]          [21, 11]          [11, 12]   
14040  14040   [Swansea, 1, Lincoln, 2]  [21, 11, 22, 11]  [11, 12, 12, 12]   

           ner_tags  sentence_len  
14037        [0, 0]             2  
14038  [3, 0, 3, 0]             4  
14039        [0, 0]             2  
14040  [3, 0, 3, 0]             4  


### Vocabulary Size

In [15]:
vocab = set()

for tokens in df_train['tokens']:
    for word in tokens:
        vocab.add(word.lower())

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 21009


<div style="border-left:6px solid #2196f3; background:#f3f9ff; padding:18px; border-radius:10px">

### 📘 SQuAD Dataset (Hugging Face)

- **Task:** Extractive Question Answering (Reading Comprehension)
- **Total Samples:** 98,169 QA pairs  
- **Splits:**
  - **Train:** 87,599 samples  
  - **Validation:** 10,570 samples  
- **Features:** `id`, `title`, `context`, `question`, `answers`  
- **Answer Type:** Text span extracted directly from context  

</div>

In [16]:
print("Loading SQuAD Dataset...")
squad = load_dataset("squad")
print(squad)

Loading SQuAD Dataset...
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [17]:
df_train = pd.DataFrame(squad['train'])
#df_val = pd.DataFrame(squad['validation'])

print(df_train.head(2), "\nShape:", df_train.shape)

                         id                     title  \
0  5733be284776f41900661182  University_of_Notre_Dame   
1  5733be284776f4190066117f  University_of_Notre_Dame   

                                             context  \
0  Architecturally, the school has a Catholic cha...   
1  Architecturally, the school has a Catholic cha...   

                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   

                                             answers  
0  {'text': ['Saint Bernadette Soubirous'], 'answ...  
1  {'text': ['a copper statue of Christ'], 'answe...   
Shape: (87599, 5)


### Sentence Lengths

In [18]:
df_train['context_len'] = df_train['context'].apply(lambda x: len(x.split()))
df_train['question_len'] = df_train['question'].apply(lambda x: len(x.split()))
df_train['answer_len'] = df_train['answers'].apply(lambda x: len(x['text'][0].split()))

print(df_train[['context_len', 'question_len', 'answer_len']].tail(10))

       context_len  question_len  answer_len
87589          144             8           3
87590          144            17           1
87591          144             9           1
87592          144            10           1
87593          144             7           2
87594          110            11           1
87595          110             6           1
87596          110             9           1
87597          110            10           1
87598          110             6           3


### Vocabulary Size

In [19]:
vocab = set()

for example in df_train.itertuples():
   # vocab.update(example.context.split())
   #vocab.update(example.question.split())
    for ans in example.answers['text']:
        vocab.update(ans.split())

# Vocabulary size
print("Vocabulary size (unique words) in training set:", len(vocab))

Vocabulary size (unique words) in training set: 55554


<div style="border-left:6px solid #4caf50; background:#f5fff7; padding:18px; border-radius:10px">
🎙️ LibriSpeech ASR Dataset (Hugging Face)

- **Task:** Automatic Speech Recognition (Speech-to-Text)
- **Total Samples:** ~28,539 audio-transcription pairs
- **Subset Used:** train-clean-100
- **Language:** English
- **Features:** `id`, `audio`, `text`, `speaker_id`, `chapter_id`  
- **Audio Type:** Clean read speech from audiobooks
- **Label Type:** Ground-truth text transcription
</div>

In [20]:
dataset = load_dataset("librispeech_asr", "clean", split="train.100")
print("Number of samples:", len(dataset))

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Error while downloading from https://huggingface.co/datasets/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/clean/train.360/0016.parquet: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/datasets/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/clean/train.360/0018.parquet: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/datasets/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/clean/train.360/0022.parquet: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/datasets/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/clean/train.360/0025.parquet: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/datasets/librispeech_asr/resolve/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1/clean/train.360/00

Generating test split:   0%|          | 0/2620 [00:00<?, ? examples/s]

Generating train.100 split:   0%|          | 0/28539 [00:00<?, ? examples/s]

Generating train.360 split:   0%|          | 0/104014 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2703 [00:00<?, ? examples/s]

Number of samples: 28539


In [25]:
pip install librosa soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 4.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.7 MB 6.3 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.7 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 4.4 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
    --------------------------------------- 0.8/38.1 MB 3.3 MB/s eta 0:00:12
   - -------------------------------------- 1.3/38.1 MB 2.8 MB/s eta 0:00:14
   - -------------------------------------- 1.6/38.1 MB 2.2 MB/s eta 0:00:17
   - -------------------------------------- 1.8/38.1 MB 1.9 MB/s eta 0:00:19
   - -------------------------------------- 1.8/38.1 MB 1.9 MB/s eta 0:00:19
   -- ------------------------------------- 2.1/38.1 MB 1.7 MB/s eta 0:00:22
   -- --------------------------

In [26]:
import librosa
import soundfile

In [27]:
print("First sample:", dataset[0])

First sample: {'file': '/home/albert/.cache/huggingface/datasets/downloads/extracted/bc0d9a6ef85c2d487c9c6efbc91f8892df927c69d3f80545a668cc058d5f677e/374-180298-0000.flac', 'audio': {'path': '374-180298-0000.flac', 'array': array([ 7.01904297e-04,  7.32421875e-04,  7.32421875e-04, ...,
       -2.74658203e-04, -1.83105469e-04, -3.05175781e-05], shape=(232480,)), 'sampling_rate': 16000}, 'text': 'CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED', 'speaker_id': 374, 'chapter_id': 180298, 'id': '374-180298-0000'}


In [28]:
print(dataset.features)

{'file': Value(dtype='string', id=None), 'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'text': Value(dtype='string', id=None), 'speaker_id': Value(dtype='int64', id=None), 'chapter_id': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}


In [29]:
print("Dataset shape:", (len(dataset), len(dataset.features)))

Dataset shape: (28539, 6)


### Label Distribution (Speaker-wise)

In [31]:
speaker_dist = {}

for example in dataset:
    spk = example["speaker_id"]
    
    if spk not in speaker_dist:
        speaker_dist[spk] = 1
    else:
        speaker_dist[spk] += 1

print("Number of speakers:", len(speaker_dist))

# Print first 10 speakers distribution
for i, (spk, count) in enumerate(speaker_dist.items()):
    print(f"Speaker {spk}: {count}")
    if i == 9:
        break

Number of speakers: 251
Speaker 374: 113
Speaker 7800: 115
Speaker 2514: 108
Speaker 3240: 127
Speaker 1088: 112
Speaker 5456: 112
Speaker 5750: 122
Speaker 1246: 117
Speaker 8238: 123
Speaker 1263: 109


### Sentence Length Distribution (Transcript Lengths)

In [32]:
sentence_lengths = {}

for example in dataset:
    text = example["text"]
    length = len(text.split())
    
    if length not in sentence_lengths:
        sentence_lengths[length] = 1
    else:
        sentence_lengths[length] += 1

print("Sentence Length Distribution (words):")

for l in sorted(sentence_lengths.keys())[:10]:
    print(f"{l} words: {sentence_lengths[l]} samples")

Sentence Length Distribution (words):
2 words: 15 samples
3 words: 21 samples
4 words: 47 samples
5 words: 94 samples
6 words: 163 samples
7 words: 219 samples
8 words: 219 samples
9 words: 234 samples
10 words: 226 samples
11 words: 237 samples


### Vocabulary Size

In [33]:
vocab = set()

for example in dataset:
    words = example["text"].lower().split()
    
    for w in words:
        vocab.add(w)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 33798
